<a href="https://colab.research.google.com/github/vasanthjilla089-cmd/blog/blob/main/vasanthakumar_Bank_Term_Deposit_Subscription_Prediction_banking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project title  

Bank Term Deposit Subscription Prediction Using Machine Learning


## Problem statement

To predict whether a bank customer will subscribe to a term deposit based on customer and campaign data using machine learning.


## **STAGE-1**  

Dateset selection with Initial EDA

UCI Machine Learning Repository – Bank Marketing Dataset

Timeline-

2014 (Dataset Published)

Location-

Portuguese Banking Institution (Portugal)

## Type of Problem

Classification

 ## Regression / Classification

Binary Classification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import zipfile
import os

# Define paths for the main zip file and its extraction directory
main_zip_path = '/content/bank+marketing (2).zip'
main_extract_dir = '/content/extracted_main_zip'

# Create the extraction directory for the main zip if it doesn't exist
os.makedirs(main_extract_dir, exist_ok=True)

# Extract the main zip file
print(f"Extracting '{main_zip_path}' to '{main_extract_dir}'...")
with zipfile.ZipFile(main_zip_path, 'r') as zip_ref:
    zip_ref.extractall(main_extract_dir)
print("Main zip extraction complete.")

# Define paths for the nested 'bank.zip' and its extraction directory
bank_zip_path = os.path.join(main_extract_dir, 'bank.zip')
bank_data_extract_dir = '/content/bank_data_csv'

# Create the extraction directory for 'bank.zip' if it doesn't exist
os.makedirs(bank_data_extract_dir, exist_ok=True)

# Extract the 'bank.zip' file
print(f"Extracting '{bank_zip_path}' to '{bank_data_extract_dir}'...")
with zipfile.ZipFile(bank_zip_path, 'r') as zip_ref:
    zip_ref.extractall(bank_data_extract_dir)
print("'bank.zip' extraction complete.")

# Now, read the 'bank-full.csv' from the final extracted location
csv_file_path = os.path.join(bank_data_extract_dir, 'bank-full.csv')

print(f"Loading data from '{csv_file_path}'...")
df = pd.read_csv(csv_file_path, sep=';') # Assuming semicolon delimiter common in bank datasets

print("DataFrame loaded successfully. Displaying the first 5 rows:")
print(df.head())

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
print(df.isnull().sum())


In [ ]:
df.count().sum()

### Shape

In [ ]:
df.shape

In [ ]:
print(df["y"].unique())

In [ ]:
print(df["y"].value_counts())

)## Describing the Dataset
Dataset Name: Bank Marketing Dataset
Source: UCI Machine Learning Repository
Total Records: 45,211
Total Features: 16 Input Features + 1 Target Variable (y)
Target Variable: y (Term Deposit Subscription: Yes / No)

### Finding the duplicates


In [ ]:
print(f"Number of duplicate rows: {df.duplicated().sum()}")

# Display the duplicate rows if any exist
if df.duplicated().sum() > 0:
    print("Duplicate rows (first 5):")
    print(df[df.duplicated()].head())

### Finding the null values

In [ ]:
df.isnull().sum()

## Target Feature
y – Term Deposit (Yes/No)
## Input Features
age
job
marital
education
default
balance
housing
loan
contact
day
month
duration
campaign
pdays
previous
poutcome

## Sample Output
y

## Information about the Dataset
Records: 45,211
Features: 16
Target: y (Yes/No)
Type: Binary Classification
Source: UCI Machine Learning Repository

## Stage 2: Data Preprocessing
Data Cleaning
Handling Missing Values
Removing Duplicates
Encoding Categorical Features
Feature and Target Selection
Train-Test Split
Feature Scaling (if required)

In [ ]:
missing_values = df.isna().sum()
print(missing_values)

# Check duplicate records


In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate Records:", duplicate_count)

In [ ]:
df = df.loc[~df.duplicated()]

In [ ]:
print("Duplicates After Cleaning:", df.duplicated().sum())

# Select numerical columns


In [ ]:
num_cols = df.select_dtypes(include=['int64', 'float64'])

In [ ]:
print(num_cols.skew())

In [ ]:
Q1 = num_cols.quantile(0.25)
Q3 = num_cols.quantile(0.75)
IQR = Q3 - Q1

outliers = ((num_cols < (Q1 - 1.5 * IQR)) | (num_cols > (Q3 + 1.5 * IQR))).sum()
print(outliers)



```
# This is formatted as code
```

### Handling skewness

In [ ]:
 print(df.skew(numeric_only=True))

In [ ]:
import numpy as np

df["balance"] = np.log1p(df["balance"])
df["duration"] = np.log1p(df["duration"])

In [ ]:
print(df[["balance", "duration"]].skew())

In [ ]:
df.describe()

In [ ]:
for col in df.select_dtypes(include='object').columns:
    print(df[col].value_counts())

In [ ]:
for  col in df.select_dtypes(include ='object').columns:
    print(df[col].value_counts())

# Target variable distribution


In [ ]:
df['y'].value_counts()

In [ ]:
df.corr(numeric_only=True)

In [ ]:
sns.heatmap(df.corr(numeric_only=True))

In [ ]:
pd.crosstab(df["job"], df["y"])

In [ ]:
X = df.drop("y", axis=1)
y = df["y"]

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Re-load df from the original CSV to ensure a clean state for this cell.
# This prevents issues from previous in-place modifications to df.
csv_file_path = '/content/bank_data_csv/bank-full.csv' # Assuming this path is correct
df = pd.read_csv(csv_file_path, sep=';')

# Apply log1p transformation only to 'duration' as it's typically non-negative and helps with skewness.
# 'balance' can contain negative values, so np.log1p is not suitable for it and caused the ValueError.
df["duration"] = np.log1p(df["duration"])

# Separate features (X) and target (y) variables
X = df.drop("y", axis=1)
y = df["y"]

# Identify categorical and numerical columns in X
numerical_cols = X.select_dtypes(include=np.number).columns
categorical_cols = X.select_dtypes(include='object').columns

# Create preprocessing pipelines for numerical and categorical features
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Create a preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Create a pipeline that first preprocesses and then trains a Logistic Regression model
model = Pipeline(steps=[('preprocessor', preprocessor),
                        ('classifier', LogisticRegression(solver='liblinear', random_state=42))])

# Convert target variable 'y' to numerical (0 for 'no', 1 for 'yes')
y_encoded = y.map({'no': 0, 'yes': 1})

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# Train the model
model.fit(X_train, y_train)

# Extract coefficients and intercept from the trained Logistic Regression model
# For binary classification, coef_ is typically shape (1, n_features) and intercept_ is (1,)
model_coffiecient = model.named_steps['classifier'].coef_
model_intercept = model.named_steps['classifier'].intercept_

m = model_coffiecient[0][0]
c = model_intercept[0]

print(f"First coefficient (m): {m}")
print(f"Intercept (c): {c}")

In [ ]:
X_values = np.linspace(X['age'].min(), X['age'].max(), 300).reshape(-1,1)

In [ ]:
def sigmoid(y):
    return 1 / (1 + np.exp(-y))

In [ ]:
y_sigmoid = sigmoid(y_encoded)

## **Stage 3**

In [ ]:
y_sigmoid

In [ ]:
X = df.drop("y", axis=1)
y = df["y"]

In [ ]:
le = LabelEncoder()

for col in df.select_dtypes('object'):
    df[col] = le.fit_transform(df[col])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# Identify categorical and numerical columns from X_train
numerical_cols = X_train.select_dtypes(include=np.number).columns
categorical_cols = X_train.select_dtypes(include='object').columns

# Create preprocessing pipelines for numerical and categorical features
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Create a preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Fit and transform X_train, and transform X_test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# The y_train and y_test are currently string labels due to prior train_test_split
# We need to explicitly convert them to numerical (0/1) for consistent model training and evaluation
y_train_encoded = y_train.map({'no': 0, 'yes': 1})
y_test_encoded = y_test.map({'no': 0, 'yes': 1})

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, solver='liblinear', random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train_processed, y_train_encoded)
    print(name, "Model Trained Successfully")

#### Defining features

In [ ]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_processed, y_train_encoded)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_processed, y_train_encoded)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_processed, y_train_encoded)

## **Stage 4**

## Stage 4: Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd # Import pandas for Series.map if needed

# Dictionary to store evaluation results
results = {}

# Evaluate Logistic Regression
y_pred_lr = lr.predict(X_test_processed)
# Ensure predictions are numerical (0 or 1) if they are currently strings ('no'/'yes')
if y_pred_lr.dtype == 'object':
    y_pred_lr = pd.Series(y_pred_lr).map({'no': 0, 'yes': 1}).values

results['Logistic Regression'] = {
    'Accuracy': accuracy_score(y_test_encoded, y_pred_lr),
    'Precision': precision_score(y_test_encoded, y_pred_lr, pos_label=1),
    'Recall': recall_score(y_test_encoded, y_pred_lr, pos_label=1),
    'F1-Score': f1_score(y_test_encoded, y_pred_lr, pos_label=1)
}

# Evaluate Decision Tree
y_pred_dt = dt.predict(X_test_processed)
if y_pred_dt.dtype == 'object':
    y_pred_dt = pd.Series(y_pred_dt).map({'no': 0, 'yes': 1}).values
results['Decision Tree'] = {
    'Accuracy': accuracy_score(y_test_encoded, y_pred_dt),
    'Precision': precision_score(y_test_encoded, y_pred_dt, pos_label=1),
    'Recall': recall_score(y_test_encoded, y_pred_dt, pos_label=1),
    'F1-Score': f1_score(y_test_encoded, y_pred_dt, pos_label=1)
}

# Evaluate Random Forest
y_pred_rf = rf.predict(X_test_processed)
if y_pred_rf.dtype == 'object':
    y_pred_rf = pd.Series(y_pred_rf).map({'no': 0, 'yes': 1}).values
results['Random Forest'] = {
    'Accuracy': accuracy_score(y_test_encoded, y_pred_rf),
    'Precision': precision_score(y_test_encoded, y_pred_rf, pos_label=1),
    'Recall': recall_score(y_test_encoded, y_pred_rf, pos_label=1),
    'F1-Score': f1_score(y_test_encoded, y_pred_rf, pos_label=1)
}

# Print results in a readable format
print("Model Evaluation Results:")
for model_name, metrics in results.items():
    print(f"\n--- {model_name} ---")
    for metric_name, value in metrics.items():
        print(f"{metric_name}: {value:.4f}")

In [ ]:
y_pred = rf.predict(X_test_processed)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test_encoded, y_pred_lr)

In [ ]:
model = LogisticRegression()

model.fit(X_train,y_train)

In [ ]:
results = {
    "Logistic Regression": accuracy_score(y_test_encoded, y_pred_lr),
    "Decision Tree": accuracy_score(y_test_encoded, y_pred_dt),
    "Random Forest": accuracy_score(y_test_encoded, y_pred_rf)
}

for model, score in results.items():
    print(f"{model}: {score:.4f}")

### GridSearchCV hyperparameter tuning

1.   List item
2.   List item



## Stage 5: Hyperparameter Tuning

#### Main optimisation code

In [ ]:
print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [ ]:
model = LogisticRegression()

model.fit(X_train,y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
accuracy_score(y_test,y_pred)

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test,y_pred)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

In [ ]:
import numpy as np
np.array(['no', 'yes'], dtype=object)

In [ ]:
df['y'].value_counts()

In [ ]:
sns.countplot(data=df, x='y')
plt.title("Yes vs No Subscription")
plt.show()